# E-commerce: hourly streaming analytics on HDFS

**Context:** cleaned invoice lines arrive in the Silver Delta directory created by notebook 08. Maintain two Gold results: hourly sales by country, and hourly quantity/sales by product.

`HDFS Silver Delta ? event-time aggregates ? foreachBatch MERGE ? HDFS Gold Delta`

Both outputs are accessed by path. Each has its own checkpoint and target directory. A changed hourly total replaces the earlier total for that key; it is not added to it again.

## Environment and storage assumptions

Examples target Spark 3.5.7 with open-source Delta Lake 3.3.2. Install matching packages in the notebook environment before creating Spark: `pip install pyspark==3.5.7 delta-spark==3.3.2`. Delta's JVM dependencies must also be available; the session helper resolves them through Maven on first use.

HDFS is available at `hdfs://localhost:9000`. This is a local Spark deployment: in a distributed deployment, `localhost` would point to each machine separately and must be replaced with a reachable NameNode hostname. HDFS paths must be readable/writable by the Spark user.

Delta is a storage format, not a cloud service. We access Delta tables by HDFS path; no Hive service, persistent metastore, or named database is needed. `DeltaCatalog` provides Spark integration and does not require Hive. Use a fresh kernel if Spark was already created without Delta extensions.

Code is supplied for explanation and adaptation; it has not been run against Spark or HDFS. Timestamp examples are timezone-free; UTC is the explicit interpretation used here. Change it to the source's documented timezone when necessary.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

# Configure Delta before creating the session; no Hive metastore is used.
builder = (SparkSession.builder.appName("EcommHDFSStreaming")
    .master("local[2]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.catalogImplementation", "in-memory")
    .config("spark.sql.shuffle.partitions", "2"))
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

## Separate the two outputs

Country and product aggregates have different schemas and keys, so they must not share a target directory. Their streaming progress must also be independent.

Silver must exist with a committed Delta schema before this notebook reads it. Notebook 08 can continue appending while these queries run. The Silver source contract is append-only; modifying/deleting old Silver rows requires a different downstream change-handling design.

In [ ]:
HDFS_ROOT = "hdfs://localhost:9000"
ECOMM_SILVER_FILES = HDFS_ROOT + "/silver/ecomm"
COUNTRY_GOLD_PATH = HDFS_ROOT + "/gold/ecomm/country_hourly_sales"
PRODUCT_GOLD_PATH = HDFS_ROOT + "/gold/ecomm/product_hourly_sales"
COUNTRY_CHECKPOINT = HDFS_ROOT + "/checkpoints/ecomm/gold_country_hourly"
PRODUCT_CHECKPOINT = HDFS_ROOT + "/checkpoints/ecomm/gold_product_hourly"

silver_df = (spark.readStream.format("delta")
    .option("maxFilesPerTrigger", 5)
    .load(ECOMM_SILVER_FILES))
silver_df.printSchema()

## Correct interpretation of watermarks

A watermark compares event timestamps with **observed event-time progress**, not today's wall clock. Data from 2010 is not automatically too old merely because the current year is later.

These examples deliberately omit a watermark so out-of-order historical files can continue revising old hourly totals. The cost is retained state for every observed country/hour and product/hour; state can grow without bound during indefinite operation.

For a live workload, choose a lateness policy and apply `withWatermark("InvoiceDate", "2 hours")` **before** grouping on InvoiceDate. That can bound window state with Update mode, but a historical backfill arriving after newer event times may then be too late. Do not enable this without deciding how late/backfill data should be handled.

## Country totals by invoice hour

A payment at 10:35 belongs to `[10:00, 11:00)`, even if Spark processes it at 12:00. HourStart is inclusive; HourEnd is exclusive.

The sum uses the decimal Amount from Silver and represents sales excluding returns. A second file containing another sale in the same hour revises that hour's total.

In [ ]:
country_hourly_sales_df = (silver_df
    .groupBy(F.window("InvoiceDate", "1 hour"), "Country")
    .agg(F.sum("Amount").alias("TotalSales"))
    .select(F.col("window.start").alias("HourStart"),
            F.col("window.end").alias("HourEnd"), "Country", "TotalSales"))

## Product totals by invoice hour

StockCode identifies the product. Sum quantities across its accepted lines, and sum Amount for its sales. Group by the product code rather than description, since descriptions may change.

The product query is a separate stateful computation, not another view of the country's aggregate state.

In [ ]:
product_hourly_sales_df = (silver_df
    .groupBy(F.window("InvoiceDate", "1 hour"), "StockCode")
    .agg(F.sum("Quantity").alias("TotalQuantity"),
         F.sum("Amount").alias("TotalSales"))
    .select(F.col("window.start").alias("HourStart"),
            F.col("window.end").alias("HourEnd"),
            "StockCode", "TotalQuantity", "TotalSales"))

## Initialize path-based Gold Delta tables

Create empty Delta tables using the aggregate schemas. This preserves Spark's inferred decimal precision for TotalSales and long integer type for TotalQuantity.

There is no SQL table registration. If a target is already Delta, leave it intact. An existing non-Delta directory causes creation to fail rather than silently overwriting it. The empty batch write initializes storage; it does not start streaming aggregation.

In [ ]:
def initialize_gold(path, schema):
    if not DeltaTable.isDeltaTable(spark, path):
        (spark.createDataFrame([], schema).write
            .format("delta").mode("errorifexists").save(path))

initialize_gold(COUNTRY_GOLD_PATH, country_hourly_sales_df.schema)
initialize_gold(PRODUCT_GOLD_PATH, product_hourly_sales_df.schema)

## Why Update mode needs an upsert

| Trigger | Current country/hour total | Correct Gold action |
|---|---|---|
| First | 100 | Insert key with 100 |
| Next | 150 | Replace its total with 150 |
| Retry of next | 150 | Leave its effective value at 150 |

The Update output contains **revised aggregate values**, not revenue deltas. Adding 150 to the existing 100 would incorrectly produce 250.

`foreachBatch` gives a bounded DataFrame for each trigger. MERGE matches by dimension and hour, updates matching rows, and inserts new keys. Delta's direct streaming sink does not support Update mode; the callback provides the upsert behavior.

In [ ]:
def merge_hourly_totals(batch_df, target_path, dimension):
    # Update-mode aggregation produces one revised row per grouping key.
    condition = (f"target.{dimension} = source.{dimension} AND "
                 "target.HourStart = source.HourStart AND "
                 "target.HourEnd = source.HourEnd")
    (DeltaTable.forPath(spark, target_path).alias("target")
        .merge(batch_df.alias("source"), condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

def write_country_batch(batch_df, batch_id):
    # Assign absolute totals; do not add them to existing target totals.
    merge_hourly_totals(batch_df, COUNTRY_GOLD_PATH, "Country")

def write_product_batch(batch_df, batch_id):
    merge_hourly_totals(batch_df, PRODUCT_GOLD_PATH, "StockCode")

## Retry behavior and ownership

`foreachBatch` may be retried. Reapplying the same deterministic key/value rows is idempotent here because MERGE assigns absolute totals. `batch_id` is provided by Spark but is not used as an arithmetic increment or a business key.

Keep one writer per Gold target and keep each query's checkpoint. This argument depends on deterministic aggregation and the append-only Silver contract; arbitrary external side effects are not made exactly once by this helper.

Deleting a checkpoint while retaining Gold is not a safe refresh procedure: state starts from scratch and partial replay totals can overwrite existing totals. A deliberate rebuild needs coordinated source, state and output handling.

## Start both queries

The one-minute trigger sets refresh cadence. The one-hour window defines invoice-time grouping; it does not make Spark wait an hour before emitting updates.

Both computations read Silver independently and use separate checkpoints. They can progress at different rates, so country and product results are not an atomic, simultaneous snapshot. If starting the second query fails, stop the first to avoid leaving a partly started pair.

In [ ]:
for query_name in ("ecomm_country_hourly", "ecomm_product_hourly"):
    if any(q.name == query_name for q in spark.streams.active):
        raise RuntimeError(f"Stop the existing query before restarting: {query_name}")

country_sales_query = (country_hourly_sales_df.writeStream
    .outputMode("update")
    .option("checkpointLocation", COUNTRY_CHECKPOINT)
    .trigger(processingTime="1 minute")
    .queryName("ecomm_country_hourly")
    .foreachBatch(write_country_batch)
    .start())

try:
    product_sales_query = (product_hourly_sales_df.writeStream
        .outputMode("update")
        .option("checkpointLocation", PRODUCT_CHECKPOINT)
        .trigger(processingTime="1 minute")
        .queryName("ecomm_product_hourly")
        .foreachBatch(write_product_batch)
        .start())
except Exception:
    country_sales_query.stop()
    raise

## Observe and read committed results

Query progress explains whether new Silver data has been processed. Read Gold with ordinary bounded DataFrames; `show()` is valid for these reads. A read taken before the first successful aggregation can return no rows.

Because the two outputs have independent progress, compare them only after both have caught up to the input of interest.

In [ ]:
print(country_sales_query.status)
print(country_sales_query.lastProgress)
print(product_sales_query.status)
print(product_sales_query.lastProgress)

# Optional bounded previews:
# spark.read.format("delta").load(COUNTRY_GOLD_PATH).orderBy("HourStart", "Country").show(10)
# spark.read.format("delta").load(PRODUCT_GOLD_PATH).orderBy("HourStart", "StockCode").show(10)

## Stop only this notebook's queries

Stopping preserves checkpoints and Gold data. A timed wait is optional and does not terminate the query on timeout. Do not stop unrelated queries sharing the SparkSession.

For scheduled catch-up, use `trigger(availableNow=True)` in place of ProcessingTime, and wait for each query to terminate. Catch up Bronze-to-Silver first if Gold must include that entire delivered batch.

In [ ]:
# Explicit lifecycle actions when needed:
# country_sales_query.stop()
# product_sales_query.stop()
# country_sales_query.awaitTermination(10)  # Timeout leaves it running.

## References

- [Spark 3.5.7 Structured Streaming guide](https://spark.apache.org/docs/3.5.7/structured-streaming-programming-guide.html)
- [Delta Lake / Spark compatibility](https://docs.delta.io/releases/)
- [Delta Lake session setup](https://docs.delta.io/quick-start/)
- [Delta streaming behavior](https://docs.delta.io/delta-streaming/)

These are adapted copies of the supplied notebooks. The originals in Downloads are unchanged. Saved outputs and platform-specific notebook metadata have been removed.